# Mini-Transformer Math Training (Resumable, Cached Data, Iterative Improvements)

This notebook trains a small Transformer for 3-digit addition with digit-by-digit supervision.

Run in order on Colab (GPU recommended):
1. Mount Drive and configure paths.
2. Build or load cached datasets from Drive.
3. Train with resumable checkpoints.
4. Evaluate and read the summary.
5. Optionally apply auto-upgrade settings and continue training.


In [ ]:
from google.colab import drive
import json
import math
import os
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# 1) Mount Google Drive
drive.mount('/content/drive')

ROOT_DIR = '/content/drive/MyDrive/math_research_v2'
DATA_DIR = os.path.join(ROOT_DIR, 'data')
CKPT_DIR = os.path.join(ROOT_DIR, 'checkpoints')
REPORT_DIR = os.path.join(ROOT_DIR, 'reports')

for d in [ROOT_DIR, DATA_DIR, CKPT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive root:', ROOT_DIR)
print('Data dir  :', DATA_DIR)
print('Ckpt dir  :', CKPT_DIR)
print('Report dir:', REPORT_DIR)


In [ ]:
# 2) Configuration
SEED = 1337
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SPECIAL_TOKENS = ['<PAD>', '<SOS>', '<EOS>']
DIGITS = list('0123456789')
SYMBOLS = list('+=|, ')
ALPHA = list('acns')
ALL_TOKENS = SPECIAL_TOKENS + DIGITS + SYMBOLS + ALPHA

stoi = {ch: i for i, ch in enumerate(ALL_TOKENS)}
itos = {i: ch for ch, i in stoi.items()}
PAD_ID = stoi['<PAD>']
SOS_ID = stoi['<SOS>']
EOS_ID = stoi['<EOS>']
VOCAB_SIZE = len(ALL_TOKENS)

# Data sizes
N_TRAIN = 120_000
N_TEST = 8_000

# Sequence length upper bound for our serialized format
BLOCK_SIZE = 64

# Training config
BATCH_SIZE = 128
MAX_ITERS = 20_000
EVAL_EVERY = 500
CKPT_EVERY = 1_000
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.05

# Model config (baseline)
D_MODEL = 192
N_HEAD = 6
N_LAYER = 6
DROPOUT = 0.15

# Iterative improvement switch
AUTO_UPGRADE_MODEL = False  # set True after reading evaluation summary


In [ ]:
# 3) Data generation and cache in Drive

def encode(text: str) -> List[int]:
    ids = []
    i = 0
    while i < len(text):
        if text.startswith('<SOS>', i):
            ids.append(SOS_ID)
            i += 5
        elif text.startswith('<EOS>', i):
            ids.append(EOS_ID)
            i += 5
        else:
            ch = text[i]
            if ch in stoi:
                ids.append(stoi[ch])
            i += 1
    return ids


def decode(ids: List[int]) -> str:
    return ''.join(itos.get(i, '') for i in ids)


def make_example(min_val: int = 1, max_val: int = 999) -> Dict[str, str]:
    a = random.randint(min_val, max_val)
    b = random.randint(min_val, max_val)

    a_rev = str(a)[::-1]
    b_rev = str(b)[::-1]

    carry = 0
    steps = []
    for i in range(max(len(a_rev), len(b_rev))):
        d1 = int(a_rev[i]) if i < len(a_rev) else 0
        d2 = int(b_rev[i]) if i < len(b_rev) else 0
        total = d1 + d2 + carry
        out_digit = total % 10
        carry = total // 10
        steps.append(f'{d1}+{d2}+c={out_digit},{carry}')

    result = str(a + b)
    cot = '|'.join(steps)
    text = f'<SOS>{a_rev}+{b_rev}= {cot}|ans={result}<EOS>'

    return {'text': text, 'a': a, 'b': b, 'answer': result}


def build_dataset(n: int, min_val: int = 1, max_val: int = 999) -> List[Dict[str, str]]:
    return [make_example(min_val=min_val, max_val=max_val) for _ in range(n)]


def save_jsonl(path: str, rows: List[Dict[str, str]]) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + '\n')


def load_jsonl(path: str) -> List[Dict[str, str]]:
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


TRAIN_PATH = os.path.join(DATA_DIR, 'train_addition_cot_v2.jsonl')
TEST_PATH = os.path.join(DATA_DIR, 'test_addition_cot_v2.jsonl')
META_PATH = os.path.join(DATA_DIR, 'dataset_meta_v2.json')

if os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH):
    print('Loading cached datasets from Drive...')
    train_rows = load_jsonl(TRAIN_PATH)
    test_rows = load_jsonl(TEST_PATH)
else:
    print('Cached dataset not found. Generating and saving...')
    train_rows = build_dataset(N_TRAIN)
    test_rows = build_dataset(N_TEST)
    save_jsonl(TRAIN_PATH, train_rows)
    save_jsonl(TEST_PATH, test_rows)
    meta = {
        'n_train': len(train_rows),
        'n_test': len(test_rows),
        'block_size': BLOCK_SIZE,
        'vocab_size': VOCAB_SIZE,
        'tokens': ALL_TOKENS,
        'seed': SEED,
    }
    with open(META_PATH, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)

print('Train samples:', len(train_rows))
print('Test samples :', len(test_rows))
print('Example text :', train_rows[0]['text'])


In [ ]:
# 4) Dataset and dataloaders
class MathDataset(Dataset):
    def __init__(self, rows: List[Dict[str, str]], block_size: int):
        self.rows = rows
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        ids = encode(self.rows[idx]['text'])
        if len(ids) < self.block_size:
            ids = ids + [PAD_ID] * (self.block_size - len(ids))
        else:
            ids = ids[:self.block_size]
            ids[-1] = EOS_ID

        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


train_ds = MathDataset(train_rows, BLOCK_SIZE)
test_ds = MathDataset(test_rows, BLOCK_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print('Train batches:', len(train_loader))
print('Test batches :', len(test_loader))


In [ ]:
# 5) Refined model architecture
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_head: int, dropout: float, block_size: int):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer('mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model: int, n_head: int, dropout: float, block_size: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MathTransformer(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, d_model: int, n_head: int, n_layer: int, dropout: float):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(d_model, n_head, dropout, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, -1),
                targets.reshape(B * T),
                ignore_index=PAD_ID,
            )
        return logits, loss


In [ ]:
# 6) Resumable training utilities
@dataclass
class ModelConfig:
    d_model: int
    n_head: int
    n_layer: int
    dropout: float
    block_size: int
    vocab_size: int


def current_model_config() -> ModelConfig:
    return ModelConfig(
        d_model=D_MODEL,
        n_head=N_HEAD,
        n_layer=N_LAYER,
        dropout=DROPOUT,
        block_size=BLOCK_SIZE,
        vocab_size=VOCAB_SIZE,
    )


def create_model(cfg: ModelConfig) -> MathTransformer:
    return MathTransformer(
        vocab_size=cfg.vocab_size,
        block_size=cfg.block_size,
        d_model=cfg.d_model,
        n_head=cfg.n_head,
        n_layer=cfg.n_layer,
        dropout=cfg.dropout,
    ).to(DEVICE)


def latest_checkpoint_path() -> str:
    best = os.path.join(CKPT_DIR, 'latest.pt')
    return best if os.path.exists(best) else ''


def save_checkpoint(path: str, model, optimizer, global_step: int, best_test_loss: float, cfg: ModelConfig):
    payload = {
        'global_step': global_step,
        'best_test_loss': best_test_loss,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'model_config': asdict(cfg),
        'rng_python': random.getstate(),
        'rng_torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        payload['rng_cuda'] = torch.cuda.get_rng_state_all()
    torch.save(payload, path)


def load_checkpoint(path: str):
    return torch.load(path, map_location=DEVICE)


@torch.no_grad()
def estimate_loss(model, loader, max_batches=50):
    model.eval()
    total = 0.0
    count = 0
    for b, (x, y) in enumerate(loader):
        if b >= max_batches:
            break
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        _, loss = model(x, y)
        total += loss.item()
        count += 1
    model.train()
    return total / max(1, count)


In [ ]:
# 7) Train (resumable)
if AUTO_UPGRADE_MODEL:
    D_MODEL = 256
    N_HEAD = 8
    N_LAYER = 8
    DROPOUT = 0.2
    LEARNING_RATE = 1.5e-4
    print('AUTO_UPGRADE_MODEL enabled: using stronger architecture.')

cfg = current_model_config()
model = create_model(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

start_step = 0
best_test_loss = float('inf')
ckpt_path = latest_checkpoint_path()

if ckpt_path:
    ckpt = load_checkpoint(ckpt_path)
    old_cfg = ckpt.get('model_config', {})
    can_resume = all(old_cfg.get(k) == getattr(cfg, k) for k in ['d_model', 'n_head', 'n_layer', 'dropout', 'block_size', 'vocab_size'])
    if can_resume:
        print('Resuming from checkpoint:', ckpt_path)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_step = int(ckpt.get('global_step', 0))
        best_test_loss = float(ckpt.get('best_test_loss', float('inf')))
        if 'rng_python' in ckpt:
            random.setstate(ckpt['rng_python'])
        if 'rng_torch' in ckpt:
            torch.set_rng_state(ckpt['rng_torch'])
        if torch.cuda.is_available() and 'rng_cuda' in ckpt:
            torch.cuda.set_rng_state_all(ckpt['rng_cuda'])
        print('Resume step:', start_step)
    else:
        print('Checkpoint config differs; starting fresh model with current config.')

loader_iter = iter(train_loader)
pbar = tqdm(range(start_step, MAX_ITERS), initial=start_step, total=MAX_ITERS)

for step in pbar:
    try:
        xb, yb = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)
        xb, yb = next(loader_iter)

    xb = xb.to(DEVICE)
    yb = yb.to(DEVICE)

    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 50 == 0:
        pbar.set_description(f'train_loss={loss.item():.4f}')

    if step > 0 and step % EVAL_EVERY == 0:
        test_loss = estimate_loss(model, test_loader)
        train_est = estimate_loss(model, train_loader, max_batches=20)
        if test_loss < best_test_loss:
            best_test_loss = test_loss
        print(f'\nstep={step} train_est={train_est:.4f} test={test_loss:.4f} best_test={best_test_loss:.4f}')

    if step > 0 and step % CKPT_EVERY == 0:
        save_checkpoint(os.path.join(CKPT_DIR, 'latest.pt'), model, optimizer, step, best_test_loss, cfg)
        save_checkpoint(os.path.join(CKPT_DIR, f'step_{step}.pt'), model, optimizer, step, best_test_loss, cfg)

save_checkpoint(os.path.join(CKPT_DIR, 'latest.pt'), model, optimizer, MAX_ITERS, best_test_loss, cfg)
torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'final_model_state.pt'))
print('Training done. Final checkpoint and model state saved.')


In [ ]:
# 8) Evaluation summary + next-step improvement suggestion
@torch.no_grad()
def greedy_generate(prompt: str, max_new_tokens: int = 64) -> str:
    model.eval()
    ids = encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        x_cond = x[:, -BLOCK_SIZE:]
        logits, _ = model(x_cond)
        next_id = int(torch.argmax(logits[:, -1, :], dim=-1).item())
        x = torch.cat([x, torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)], dim=1)
        if next_id == EOS_ID:
            break
    return decode(x[0].tolist())


def parse_predicted_answer(text: str) -> str:
    marker = 'ans='
    if marker not in text:
        return ''
    tail = text.split(marker, 1)[1]
    ans = []
    for ch in tail:
        if ch.isdigit():
            ans.append(ch)
        else:
            break
    return ''.join(ans)


@torch.no_grad()
def evaluate_exact_match(n_cases: int = 200) -> Dict[str, float]:
    correct = 0
    samples = []
    for _ in range(n_cases):
        ex = make_example()
        prompt = f"<SOS>{str(ex['a'])[::-1]}+{str(ex['b'])[::-1]}="
        out = greedy_generate(prompt)
        pred = parse_predicted_answer(out)
        ok = pred == ex['answer']
        correct += int(ok)
        if len(samples) < 5:
            samples.append({'a': ex['a'], 'b': ex['b'], 'gold': ex['answer'], 'pred': pred, 'output': out, 'ok': ok})

    acc = correct / n_cases
    return {'exact_match': acc, 'num_cases': n_cases, 'examples': samples}


metrics = evaluate_exact_match(n_cases=300)
train_loss_est = estimate_loss(model, train_loader, max_batches=30)
test_loss_est = estimate_loss(model, test_loader, max_batches=30)

summary = {
    'train_loss_est': train_loss_est,
    'test_loss_est': test_loss_est,
    'exact_match': metrics['exact_match'],
    'num_eval_cases': metrics['num_cases'],
    'model_config': asdict(cfg),
    'best_test_loss': best_test_loss,
}

summary_path = os.path.join(REPORT_DIR, 'eval_summary_v2.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Evaluation Summary')
print(json.dumps(summary, indent=2))
print('Saved summary to:', summary_path)
print('Sample predictions:')
for s in metrics['examples']:
    print(s)

print('\nImprovement policy:')
if summary['exact_match'] < 0.70:
    print('- Accuracy is low. Increase model size (AUTO_UPGRADE_MODEL=True) and continue training.')
elif summary['exact_match'] < 0.90:
    print('- Accuracy is moderate. Train longer and reduce learning rate by 20-30%.')
else:
    print('- Accuracy is strong. Focus on harder cases (4-digit addition) or compression.')
